# 13 — Portfolio VaR and Expected Shortfall

This post-protocol ancillary notebook compares CA-RNN with both the likelihood-only RNN and the VAR–GARCH joint-residual bootstrap as downstream portfolio-risk models. It uses retained forecasts, so no model is retrained. Standardized component returns are converted to an equal-weight, unitless loss index; this is a model diagnostic, not the profit and loss of an investable portfolio. The primary tail is the prespecified 5% upper loss tail (95% VaR).

Expected Shortfall here is a risk measure, not the Energy Score used elsewhere. VaR is assessed with exceedance diagnostics and quantile loss; VaR and Expected Shortfall are assessed jointly with the FZ0 proper score. Lower scores are better. Coverage-test p-values are descriptive under serial dependence; paired conclusions use the hierarchical moving-block bootstrap.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from innovcal.ca_rnn.evaluation import (
    paired_matched_seed_tail_risk_bootstrap,
    paired_seed_tail_risk_bootstrap,
    portfolio_tail_forecasts,
    summarize_tail_risk,
)

warnings.filterwarnings('ignore', category=RuntimeWarning)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'results' / 'tail_risk'
OUT.mkdir(parents=True, exist_ok=True)
TAIL_PROBABILITY = 0.05
BLOCK_LENGTH = 20
N_BOOTSTRAP = 2000

In [ ]:
seed_archive = np.load(ROOT / 'results' / 'seed_uncertainty' / 'seed_forecasts.npz')
seed_manifest = pd.read_csv(ROOT / 'results' / 'seed_uncertainty' / 'seed_forecast_manifest.csv')
baseline_archive = np.load(ROOT / 'results' / 'conventional_baselines' / 'baseline_forecasts.npz')

# The archived manifest predates the final naming convention. Its full
# projected-plus-temporal specification is the model now called CA-RNN.
archived_full_name = 'CA-RNN sequential' if 'CA-RNN sequential' in set(seed_manifest.model) else 'CA-RNN'
ca_key = seed_manifest.loc[seed_manifest.model == archived_full_name, 'array_key'].iloc[0]
rnn_key = seed_manifest.loc[seed_manifest.model == 'RNN', 'array_key'].iloc[0]
target = seed_archive['target'].astype(float)
ca_samples = seed_archive[ca_key].astype(float)
rnn_samples = seed_archive[rnn_key].astype(float)
var_garch_samples = baseline_archive['VAR_GARCH_bootstrap'].astype(float)
np.testing.assert_allclose(target, baseline_archive['target'], atol=1e-6)
weights = np.full(target.shape[1], 1.0 / target.shape[1])
print(f'{ca_samples.shape[0]} CA-RNN seeds; {len(target)} matched origins; {ca_samples.shape[1]} draws per seed')
print('Equal portfolio weights:', weights)

In [ ]:
rows = []
for model, model_samples in [('RNN', rnn_samples), ('CA-RNN', ca_samples)]:
    for seed_index, samples in enumerate(model_samples):
        rows.append({'model': model, 'training_seed_index': seed_index, **summarize_tail_risk(
            target, samples, tail_probability=TAIL_PROBABILITY, weights=weights
        )})
rows.append({'model': 'VAR-GARCH bootstrap', 'training_seed_index': np.nan, **summarize_tail_risk(
    target, var_garch_samples, tail_probability=TAIL_PROBABILITY, weights=weights
)})
seed_metrics = pd.DataFrame(rows)
seed_metrics.to_csv(OUT / 'portfolio_tail_risk_by_seed.csv', index=False)
display(seed_metrics)

summary = seed_metrics.groupby('model', sort=False).agg(
    exceedance_rate=('exceedance_rate', 'mean'),
    exceedance_rate_sd=('exceedance_rate', 'std'),
    mean_value_at_risk=('mean_value_at_risk', 'mean'),
    mean_expected_shortfall=('mean_expected_shortfall', 'mean'),
    mean_exceedance_loss=('mean_exceedance_loss', 'mean'),
    mean_quantile_loss=('mean_quantile_loss', 'mean'),
    mean_joint_var_es_score=('mean_joint_var_es_score', 'mean'),
).reset_index()
summary.to_csv(OUT / 'portfolio_tail_risk_summary.csv', index=False)
display(summary)

In [ ]:
ca_vs_rnn = paired_matched_seed_tail_risk_bootstrap(
    target, ca_samples, rnn_samples,
    candidate_name='CA-RNN', comparator_name='RNN',
    tail_probability=TAIL_PROBABILITY, weights=weights,
    block_length=BLOCK_LENGTH, n_bootstrap=N_BOOTSTRAP, n_draw_resamples=200,
    seed=13_013, draw_seed=13_113, familywise_confidence=0.975,
)
ca_vs_vg = paired_seed_tail_risk_bootstrap(
    target,
    ca_samples,
    var_garch_samples,
    tail_probability=TAIL_PROBABILITY,
    weights=weights,
    block_length=BLOCK_LENGTH,
    n_bootstrap=N_BOOTSTRAP,
    n_draw_resamples=200,
    seed=13_014,
    draw_seed=13_114,
    familywise_confidence=0.975,
)
inference = pd.concat([ca_vs_rnn, ca_vs_vg], ignore_index=True)
inference['comparison_family_count'] = 2
inference['outer_bonferroni_confidence'] = 0.975
inference.to_csv(OUT / 'portfolio_tail_risk_inference.csv', index=False)
display(inference)
print('Differences are CA-RNN minus comparator; negative proper-score differences favor CA-RNN.')
print('Intervals use max-|z| across scores and Bonferroni across the two comparisons.')

In [ ]:
ca_paths = [portfolio_tail_forecasts(target, value, tail_probability=TAIL_PROBABILITY, weights=weights) for value in ca_samples]
vg_path = portfolio_tail_forecasts(target, var_garch_samples, tail_probability=TAIL_PROBABILITY, weights=weights)
ca_var = np.mean([frame.value_at_risk.to_numpy() for frame in ca_paths], axis=0)
ca_es = np.mean([frame.expected_shortfall.to_numpy() for frame in ca_paths], axis=0)
loss = vg_path.realized_loss.to_numpy()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(loss, color='black', linewidth=0.7, label='Realized loss')
axes[0].plot(ca_var, label='CA-RNN VaR', alpha=0.9)
axes[0].plot(vg_path.value_at_risk, label='VAR-GARCH VaR', alpha=0.9)
axes[0].plot(ca_es, '--', label='CA-RNN Expected Shortfall', alpha=0.8)
axes[0].plot(vg_path.expected_shortfall, '--', label='VAR-GARCH Expected Shortfall', alpha=0.8)
axes[0].set_ylabel('Standardized portfolio loss')
axes[0].legend(ncol=2, fontsize=8)

expected = TAIL_PROBABILITY * np.arange(1, len(target) + 1)
ca_hits = np.mean([np.cumsum(frame.exceedance) for frame in ca_paths], axis=0)
axes[1].plot(expected, color='black', linestyle=':', label='Nominal expected count')
axes[1].plot(ca_hits, label='CA-RNN mean cumulative exceedances')
axes[1].plot(np.cumsum(vg_path.exceedance), label='VAR-GARCH cumulative exceedances')
axes[1].set_xlabel('Forecast origin')
axes[1].set_ylabel('Cumulative exceedances')
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / 'portfolio_tail_risk_paths.png', dpi=180, bbox_inches='tight')
plt.show()

## Interpretation rules

A VaR exceedance rate near 0.05 is necessary but not sufficient: inspect clustering and quantile loss as well. Lower FZ0 joint score indicates a better VaR–Expected Shortfall pair. Treat the Kupiec and independence p-values as descriptive because daily forecast errors are dependent and the models are estimated. CA-RNN versus RNN uses matched optimization seeds, shared date blocks, and paired predictive draws; CA-RNN versus VAR–GARCH uses shared date blocks and independent predictive-draw resampling. Max-|z| intervals cover the two scores, with Bonferroni allocation across the two model comparisons. The audit still conditions on the fitted models and remains post-protocol ancillary analysis.